In [5]:

import pandas as pd
import numpy as np



In [6]:
# Load dataset
df = pd.read_csv(r"C:\Users\Renu Sharma\Downloads\healthcare_data_cleaning_dataset.csv")

In [9]:
# Q1. Missing Data Identification

# Check missing values
missing_values = df.isnull().sum()

# Calculate percentage of missing values
missing_percentage = (df.isnull().sum() / len(df)) * 100

# Create summary table
missing_data = pd.DataFrame({
    "Missing Values": missing_values,
    "Percentage": missing_percentage
})

print(missing_data)

                    Missing Values  Percentage
Patient_ID                       0    0.000000
Age                            600   11.764706
Gender                           0    0.000000
City                             0    0.000000
Diagnosis                        0    0.000000
Hospital_Visits                  0    0.000000
Treatment_Cost                 593   11.627451
Insurance_Coverage               0    0.000000
Admission_Date                   0    0.000000


In [15]:
# Q2. Handling Missing Age

# Check missing values in Age column
print(df["Age"].isnull().sum())

# Replace missing Age values with median
median_age = df["Age"].median()
df["Age"].fillna(median_age, inplace=True)



600


C:\Users\Renu Sharma\AppData\Local\Temp\ipykernel_17152\1158907932.py:8: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df["Age"].fillna(median_age, inplace=True)


0       35.0
1       21.0
2       77.0
3       79.0
4       60.0
        ... 
5095    50.0
5096    50.0
5097    50.0
5098    50.0
5099    50.0
Name: Age, Length: 5100, dtype: float64

In [16]:
# Q3. Handling Missing Treatment Cost

# Check missing values in Treatment_Cost
print(df["Treatment_Cost"].isnull().sum())

# Replace missing Treatment_Cost values with median
median_cost = df["Treatment_Cost"].median()

df["Treatment_Cost"].fillna(median_cost, inplace=True)

# Verify again
print(df["Treatment_Cost"].isnull().sum())



593
593


C:\Users\Renu Sharma\AppData\Local\Temp\ipykernel_17152\62933841.py:9: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df["Treatment_Cost"].fillna(median_cost, inplace=True)


In [17]:
# Q4. Duplicate Patient Records

# Dataset size before removing duplicates
before_rows = df.shape[0]

print("Rows before removing duplicates:", before_rows)

# Identify duplicate rows
duplicates = df[df.duplicated()]

print("Duplicate Rows:")
print(duplicates)

# Remove duplicate rows
df = df.drop_duplicates()

# Dataset size after removing duplicates
after_rows = df.shape[0]

print("Rows after removing duplicates:", after_rows)

# Compare dataset size
print("Total duplicates removed:", before_rows - after_rows)

Rows before removing duplicates: 5100
Duplicate Rows:
      Patient_ID  Age  Gender       City     Diagnosis  Hospital_Visits  \
5000       15126  NaN    Male  Hyderabad        Asthma                1   
5001       15108  NaN    Male  Hyderabad        Asthma                3   
5002       10441  NaN    Male    Chennai           Flu                3   
5003       15975  NaN  Female  Hyderabad           Flu               10   
5004       18427  NaN  Female      Delhi      COVID-19               13   
...          ...  ...     ...        ...           ...              ...   
5094       11571  NaN  Female    Chennai      COVID-19                4   
5096       17597  NaN  Female    Chennai        Asthma                2   
5097       19171  NaN  Female     Mumbai           Flu                1   
5098       13854  NaN  Female  Bangalore           Flu               17   
5099       14859  NaN    Male      Delhi  Hypertension               19   

      Treatment_Cost  Insurance_Coverage Admi

In [18]:
# Q5. Invalid Age Values (Data Quality Check)

# Detect invalid age values (<0 or >100)
invalid_age = df[(df["Age"] < 0) | (df["Age"] > 100)]

print("Invalid Age Records:")
print(invalid_age)

# Count invalid records
print("Number of invalid age values:", invalid_age.shape[0])

# Option 1: Remove invalid records
df = df[(df["Age"] >= 0) & (df["Age"] <= 100)]

Invalid Age Records:
Empty DataFrame
Columns: [Patient_ID, Age, Gender, City, Diagnosis, Hospital_Visits, Treatment_Cost, Insurance_Coverage, Admission_Date]
Index: []
Number of invalid age values: 0


In [19]:
# Q6. Outlier Detection (Treatment Cost)

# Calculate Q1 and Q3
Q1 = df["Treatment_Cost"].quantile(0.25)
Q3 = df["Treatment_Cost"].quantile(0.75)

# Interquartile Range (IQR)
IQR = Q3 - Q1

# Define outlier boundaries
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Detect outliers
outliers = df[(df["Treatment_Cost"] < lower_bound) | (df["Treatment_Cost"] > upper_bound)]

print("Number of outliers in Treatment_Cost:", outliers.shape[0])

Number of outliers in Treatment_Cost: 43


In [20]:
# Q7. Outlier Treatment

# Calculate 5th and 95th percentile
lower_percentile = df["Treatment_Cost"].quantile(0.05)
upper_percentile = df["Treatment_Cost"].quantile(0.95)

# Apply capping (Winsorization)
df["Treatment_Cost_Capped"] = df["Treatment_Cost"].copy()

df["Treatment_Cost_Capped"] = df["Treatment_Cost_Capped"].apply(
    lambda x: lower_percentile if x < lower_percentile 
    else upper_percentile if x > upper_percentile 
    else x
)

# Check result
print(df[["Treatment_Cost", "Treatment_Cost_Capped"]].head())

   Treatment_Cost  Treatment_Cost_Capped
0         41010.0                41010.0
1         12194.0                12194.0
2         45086.0                45086.0
3         40842.0                40842.0
4          9873.0                 9873.0


In [21]:
# 8. Transformation


# Create new column with log transformation
df["Treatment_Cost_Log"] = np.log1p(df["Treatment_Cost"])  
# log1p = log(1 + x) to handle zero values safely

# Compare original vs transformed values
print(df[["Treatment_Cost", "Treatment_Cost_Log"]].head())

   Treatment_Cost  Treatment_Cost_Log
0         41010.0           10.621596
1         12194.0            9.408781
2         45086.0           10.716349
3         40842.0           10.617491
4          9873.0            9.197660


In [28]:
# Q9. Time-Based Missing Handling

import pandas as pd

# Ensure Admission_Date is in datetime format
df["Admission_Date"] = pd.to_datetime(df["Admission_Date"])

# Sort data by Admission_Date
df = df.sort_values(by="Admission_Date")

# Apply forward fill (or backward fill if needed)
df["Admission_Date"] = df["Admission_Date"].ffill()   # forward fill
# Alternative:
# df["Admission_Date"] = df["Admission_Date"].bfill()  # backward fill

# Check missing values
print(df["Admission_Date"].isnull().sum())

# Show first few rows to confirm changes
print(df.head(10))


0
      Patient_ID   Age  Gender       City     Diagnosis  Hospital_Visits  \
4329       17692  64.0    Male    Chennai  Hypertension               19   
2932       19591  91.0    Male    Chennai        Asthma                3   
4617       10896  88.0  Female    Chennai      COVID-19                3   
2624       16450  57.0  Female    Chennai      Diabetes               15   
2702       17230  41.0    Male  Bangalore  Hypertension               18   
866        15966  49.0  Female     Mumbai  Hypertension               10   
4563       19175  12.0  Female      Delhi           Flu                3   
4837       12843  89.0  Female  Bangalore      COVID-19               14   
2838       15443   4.0  Female  Hyderabad  Hypertension               18   
348        18716  81.0    Male      Delhi           Flu                4   

      Treatment_Cost  Insurance_Coverage Admission_Date  \
4329    36377.000000                   1     2023-01-01   
2932    32149.000000                   0   